# Silver: POS events
- typed columns + `arrival_date` / `is_late` from the landing folder
- bad rows -> `pos_quarantine` with a reason (never silently dropped)
- dedup **within** a batch (`row_number`, earliest arrival wins) and **across** batches (`MERGE ... whenNotMatchedInsert`)
- `txnAppId` + `txnVersion` make a retried `foreachBatch` idempotent

In [0]:
%run ./_common

In [0]:
NB = "20_silver_pos"
SRC, TGT, QUAR = f"{CAT}.bronze.pos_events", f"{CAT}.silver.pos_events", f"{CAT}.silver.pos_quarantine"
CP = f"{CKPT}/silver_pos"

cols = """event_id STRING, event_ts TIMESTAMP, event_date DATE, store_id STRING, item_id STRING,
          units INT, unit_price DECIMAL(10,2), arrival_date DATE, is_late BOOLEAN,
          _source_file STRING, _ingested_at TIMESTAMP"""
spark.sql(f"CREATE TABLE IF NOT EXISTS {TGT} ({cols}, _silver_at TIMESTAMP) CLUSTER BY (event_date, store_id)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {QUAR} ({cols}, _rescued_data STRING, _dq_reason STRING, _quarantined_at TIMESTAMP)")

def clean(df):
    arrival = F.to_date(F.regexp_extract("_source_file", r"pos/(\d{4}-\d{2}-\d{2})/", 1))
    out = df.select(
        "event_id",
        F.to_timestamp("event_ts").alias("event_ts"),
        F.to_date("event_date").alias("event_date"),
        "store_id", "item_id",
        F.col("units").cast("int").alias("units"),
        F.col("unit_price").cast("decimal(10,2)").alias("unit_price"),
        arrival.alias("arrival_date"),
        "_source_file", "_ingested_at", "_rescued_data",
    ).withColumn("is_late", F.col("arrival_date") > F.col("event_date"))
    reason = (F.when(F.col("event_id").isNull(), "missing_event_id")
               .when(F.col("_rescued_data").isNotNull(), "rescued_data")
               .when(F.col("event_date").isNull(), "invalid_event_date")
               .when(F.col("units").isNull() | (F.col("units") <= 0), "invalid_units"))
    return out.withColumn("_dq_reason", reason)

def process(batch_df, batch_id):
    s = batch_df.sparkSession
    df = clean(batch_df)
    (df.filter("_dq_reason IS NOT NULL").withColumn("_quarantined_at", F.current_timestamp())
       .select(*s.table(QUAR).columns).write.mode("append")
       .option("txnAppId", "silver_pos_quarantine").option("txnVersion", batch_id)
       .saveAsTable(QUAR))
    first = Window.partitionBy("event_id").orderBy("arrival_date", "_source_file")
    good = (df.filter("_dq_reason IS NULL")
              .withColumn("_rn", F.row_number().over(first)).filter("_rn = 1")
              .withColumn("_silver_at", F.current_timestamp())
              .select(*s.table(TGT).columns))
    (DeltaTable.forName(s, TGT).alias("t")
        .merge(good.alias("s"), "t.event_date = s.event_date AND t.event_id = s.event_id")
        .whenNotMatchedInsertAll().execute())

before = table_count(TGT)
q = (spark.readStream.option("maxBytesPerTrigger", "256m").table(SRC)
       .writeStream.foreachBatch(process)
       .option("checkpointLocation", CP)
       .trigger(availableNow=True).start())
q.awaitTermination()
print(f"{table_count(TGT) - before:,} new events in Silver (total {table_count(TGT):,})")

In [0]:
r = spark.sql(f"""
  SELECT (SELECT COUNT(DISTINCT event_id) FROM {SRC})  AS bronze_ids,
         (SELECT COUNT(*) FROM {TGT})                  AS silver_rows,
         (SELECT COUNT(DISTINCT event_id) FROM {TGT})  AS silver_ids,
         (SELECT COUNT(DISTINCT event_id) FROM {QUAR}) AS quarantined_ids""").first()
dq_check("silver_pos_duplicates", r.silver_rows - r.silver_ids, r.silver_rows == r.silver_ids, "0")
dq_check("silver_pos_reconciles_to_bronze", r.silver_ids + r.quarantined_ids,
         r.silver_ids + r.quarantined_ids == r.bronze_ids, f"{r.bronze_ids} (bronze distinct ids)")
dq_finish(NB)